# 02 · Similarity Search
### Practical LLM Engineering — Module 03: Embeddings & Retrieval

> **What you'll learn**
> - Exact nearest-neighbour search with brute-force cosine/dot/L2
> - Approximate Nearest Neighbour (ANN): IVF, HNSW, LSH
> - Building and benchmarking a FAISS index from scratch
> - Recall@k, latency, and the ANN accuracy/speed tradeoff
> - Engineering patterns: index selection, batch querying, ID mapping

---


## 1. Overview

Given a query embedding $\mathbf{q} \in \mathbb{R}^d$ and a corpus of $N$ vectors $\{\mathbf{x}_i\}_{i=1}^N$, the **nearest-neighbour search** problem is:

$$
\text{NN}(\mathbf{q}) = \underset{i}{\arg\max}\; \text{sim}(\mathbf{q}, \mathbf{x}_i)
$$

- **Exact search** (brute force): $\mathcal{O}(Nd)$ — correct but slow for large $N$
- **Approximate search** (ANN): $\mathcal{O}(\log N \cdot d)$ — tiny accuracy loss, huge speed gain

### When exact vs ANN

| Corpus size | Method | Why |
|---|---|---|
| < 10K vectors | Brute force numpy | Fast enough, no index overhead |
| 10K–1M vectors | FAISS IVF or HNSW | Significant speedup, ~1% recall loss |
| > 1M vectors | FAISS IVFPQ or ScaNN | Compression + partitioning essential |

### Module position

```
01_sentence_embeddings   ✅
02_similarity_search     ← you are here
03_vector_databases
04_chunking_strategies
05_hybrid_search
```


## 2. Setup

In [ ]:
# !pip install faiss-cpu numpy matplotlib scikit-learn -q
import time, random, math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize

print("✅ Libraries ready")

# ── Tiny embedding client reused from notebook 01 ─────────────────────
class MockEmbedder:
    def __init__(self, dim=64, seed=42):
        self.dim  = dim
        self._proj = np.random.RandomState(seed).randn(10_000, dim).astype(np.float32)
    def _hash(self, w): return int(abs(hash(w.lower())) % 10_000)
    def embed_one(self, text):
        tokens = text.lower().split()
        if not tokens: return np.zeros(self.dim, dtype=np.float32)
        v = np.mean([self._proj[self._hash(t)] for t in tokens], axis=0)
        n = np.linalg.norm(v); return v / max(n, 1e-9)
    def embed(self, texts):
        return normalize(np.array([self.embed_one(t) for t in texts], dtype=np.float32))

embedder = MockEmbedder(dim=64)
print(f"MockEmbedder dim={embedder.dim}")


## 3. Brute-Force Exact Search

Exact search computes the full similarity matrix:

$$
S_{ij} = \mathbf{q}_i \cdot \mathbf{x}_j \quad \forall i,j
$$

With L2-normalised vectors this equals cosine similarity. For $N$ corpus vectors and $k$ queries, cost is $\mathcal{O}(kNd)$.


In [ ]:
def brute_force_search(query_vecs: np.ndarray,
                        corpus_vecs: np.ndarray,
                        k: int = 5) -> tuple[np.ndarray, np.ndarray]:
    """
    Exact cosine nearest-neighbour search via batch dot product.
    Assumes both query_vecs and corpus_vecs are L2-normalised.
    Returns: (indices, scores) each shape (n_queries, k)
    """
    # (n_queries, n_corpus)
    scores = query_vecs @ corpus_vecs.T
    # Top-k per query
    top_k_idx = np.argsort(-scores, axis=1)[:, :k]
    top_k_scores = np.take_along_axis(scores, top_k_idx, axis=1)
    return top_k_idx, top_k_scores


# ── Build a demo corpus ───────────────────────────────────────────────
TOPICS = ["machine learning", "deep learning", "neural networks", "transformers",
           "cooking recipes", "french cuisine", "italian pasta", "baking bread",
           "quantum physics", "relativity theory", "particle accelerators",
           "history of rome", "roman empire", "ancient greece", "medieval europe"]

corpus_texts = []
for topic in TOPICS:
    for i in range(20):
        corpus_texts.append(f"{topic} concept number {i} with some detail")

corpus_vecs = embedder.embed(corpus_texts)
print(f"Corpus: {len(corpus_texts)} texts, shape={corpus_vecs.shape}")

# ── Query ──────────────────────────────────────────────────────────────
queries = [
    "introduction to deep learning and neural networks",
    "how to make pasta carbonara at home",
    "Einstein general theory of relativity spacetime",
]
query_vecs = embedder.embed(queries)

t0 = time.perf_counter()
indices, scores = brute_force_search(query_vecs, corpus_vecs, k=3)
ms = (time.perf_counter() - t0) * 1000

print(f"\nBrute-force search: {len(queries)} queries → {ms:.2f}ms\n")
for i, (q, idxs, scs) in enumerate(zip(queries, indices, scores)):
    print(f"Q{i+1}: {q}")
    for rank, (idx, sc) in enumerate(zip(idxs, scs), 1):
        print(f"  #{rank} [{sc:.3f}] {corpus_texts[idx][:55]}")
    print()


## 4. Approximate Nearest Neighbour (ANN)

### 4.1 Inverted File Index (IVF)

IVF partitions the vector space into $n_{\text{list}}$ Voronoi cells using k-means. At query time, only the $n_{\text{probe}}$ nearest cells are searched:

```
Training: k-means → n_list centroids
Indexing: assign each vector to nearest centroid
Querying: find top n_probe centroids → search only those cells
```

Tradeoff: larger $n_{\text{probe}}$ → higher recall, higher latency.

### 4.2 Hierarchical Navigable Small Worlds (HNSW)

HNSW builds a multi-layer proximity graph. Queries traverse from coarse to fine layers:

```
Layer 2 (coarse): few long-range connections
Layer 1 (medium): medium connections
Layer 0 (fine):   dense local connections ← final k-NN candidates here
```

HNSW has $\mathcal{O}(\log N)$ query time with excellent recall/speed tradeoffs — it is the default in most production vector databases.

### 4.3 FAISS

Facebook AI Similarity Search (FAISS) is the standard library for ANN. Key index types:

| Index | Description | Best for |
|---|---|---|
| `IndexFlatL2` | Exact L2 | < 100K, ground truth |
| `IndexFlatIP` | Exact inner product | < 100K, normalised vecs |
| `IndexIVFFlat` | IVF + exact within cells | 100K–10M |
| `IndexHNSWFlat` | HNSW graph | Any size, best recall/speed |
| `IndexIVFPQ` | IVF + product quantisation | > 10M, memory-constrained |


In [ ]:
try:
    import faiss
    FAISS_AVAILABLE = True
    print(f"✅ FAISS {faiss.__version__} available")
except ImportError:
    FAISS_AVAILABLE = False
    print("⚠️  FAISS not installed. Run: pip install faiss-cpu")
    print("    Falling back to numpy brute-force for all demos.")


# ── FAISS index factory ───────────────────────────────────────────────
class VectorIndex:
    """
    Thin wrapper around FAISS with a numpy fallback.
    Supports: flat (exact), IVF, HNSW.
    """
    def __init__(self, dim: int, index_type: str = "flat",
                 n_list: int = 32, n_probe: int = 8,
                 hnsw_m: int = 32):
        self.dim        = dim
        self.index_type = index_type
        self.n_list     = n_list
        self.n_probe    = n_probe
        self._ids       = []   # parallel list: internal FAISS id → external id

        if FAISS_AVAILABLE:
            if index_type == "flat":
                self._idx = faiss.IndexFlatIP(dim)
            elif index_type == "ivf":
                quantiser = faiss.IndexFlatIP(dim)
                self._idx = faiss.IndexIVFFlat(quantiser, dim, n_list,
                                                faiss.METRIC_INNER_PRODUCT)
            elif index_type == "hnsw":
                self._idx = faiss.IndexHNSWFlat(dim, hnsw_m,
                                                  faiss.METRIC_INNER_PRODUCT)
            else:
                raise ValueError(f"Unknown index_type: {index_type!r}")
        else:
            self._idx  = None
            self._vecs = np.zeros((0, dim), dtype=np.float32)

    def train(self, vecs: np.ndarray):
        """Train IVF index (no-op for flat/HNSW)."""
        if FAISS_AVAILABLE and self.index_type == "ivf":
            self._idx.train(vecs.astype(np.float32))

    def add(self, vecs: np.ndarray, ids: list = None):
        """Add vectors to the index."""
        vecs = vecs.astype(np.float32)
        if ids is None:
            ids = list(range(len(self._ids), len(self._ids) + len(vecs)))
        self._ids.extend(ids)
        if FAISS_AVAILABLE:
            self._idx.add(vecs)
        else:
            self._vecs = np.vstack([self._vecs, vecs]) if len(self._vecs) else vecs

    def search(self, query_vecs: np.ndarray, k: int = 5) -> tuple[np.ndarray, np.ndarray]:
        """Return (external_ids, scores) arrays of shape (n_queries, k)."""
        query_vecs = query_vecs.astype(np.float32)
        if FAISS_AVAILABLE:
            if self.index_type == "ivf":
                self._idx.nprobe = self.n_probe
            scores, internal_ids = self._idx.search(query_vecs, k)
            # Map internal → external IDs
            ext_ids = np.array([[self._ids[j] if j >= 0 else -1
                                  for j in row] for row in internal_ids])
        else:
            scores_np = query_vecs @ self._vecs.T
            internal_ids = np.argsort(-scores_np, axis=1)[:, :k]
            scores = np.take_along_axis(scores_np, internal_ids, axis=1)
            ext_ids = np.array([[self._ids[j] for j in row]
                                  for row in internal_ids])
        return ext_ids, scores

    @property
    def ntotal(self): 
        if FAISS_AVAILABLE: return self._idx.ntotal
        return len(self._vecs)


# ── Build and query three index types ─────────────────────────────────
index_configs = [
    ("flat",  {"index_type": "flat"}),
    ("ivf",   {"index_type": "ivf",  "n_list": 8, "n_probe": 4}),
    ("hnsw",  {"index_type": "hnsw", "hnsw_m": 16}),
]

print("Index comparison on 300-vector corpus:")
print(f"{'Index':>8} {'Build(ms)':>10} {'Query(ms)':>10} {'Recall@3':>10}")
print("-" * 44)

# Ground truth from brute force
gt_ids, _ = brute_force_search(query_vecs, corpus_vecs, k=3)

for name, cfg in index_configs:
    idx = VectorIndex(dim=embedder.dim, **cfg)
    t0  = time.perf_counter()
    idx.train(corpus_vecs)
    idx.add(corpus_vecs)
    build_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    ids, scores = idx.search(query_vecs, k=3)
    query_ms = (time.perf_counter() - t0) * 1000

    # Recall@3: fraction of GT neighbours found
    recall = np.mean([
        len(set(ids[i]) & set(gt_ids[i])) / 3
        for i in range(len(query_vecs))
    ])
    print(f"{name:>8} {build_ms:>10.1f} {query_ms:>10.2f} {recall:>10.3f}")


## 5. ANN Accuracy/Speed Tradeoff

In [ ]:
# ── nprobe sweep for IVF index ────────────────────────────────────────
# Build a larger corpus for a more meaningful sweep
N_LARGE = 2000
np.random.seed(0)
large_vecs = normalize(np.random.randn(N_LARGE, embedder.dim).astype(np.float32))
large_ids  = list(range(N_LARGE))

# Build exact index for ground truth
gt_idx = VectorIndex(dim=embedder.dim, index_type="flat")
gt_idx.add(large_vecs, large_ids)

# Build IVF with 50 cells
n_list = 50
ivf_idx = VectorIndex(dim=embedder.dim, index_type="ivf", n_list=n_list, n_probe=1)
ivf_idx.train(large_vecs)
ivf_idx.add(large_vecs, large_ids)

# Test queries
n_q = 50
q_vecs = normalize(np.random.randn(n_q, embedder.dim).astype(np.float32))
k = 10

gt_results, _ = gt_idx.search(q_vecs, k=k)

nprobe_values = [1, 2, 4, 8, 16, 25, 50]
recalls, latencies = [], []

for nprobe in nprobe_values:
    ivf_idx.n_probe = nprobe
    t0 = time.perf_counter()
    for _ in range(5):   # repeat for stable timing
        ann_results, _ = ivf_idx.search(q_vecs, k=k)
    ms = (time.perf_counter() - t0) * 1000 / 5

    recall = np.mean([
        len(set(ann_results[i]) & set(gt_results[i])) / k
        for i in range(n_q)
    ])
    recalls.append(recall)
    latencies.append(ms)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle(f"IVF nprobe sweep (n_list={n_list}, N={N_LARGE}, k={k})", fontsize=11)

ax1.plot(nprobe_values, recalls, "o-", color="#3498db", lw=2, markersize=7)
ax1.axhline(1.0, color="gray", linestyle="--", alpha=0.5)
ax1.set_xlabel("nprobe"); ax1.set_ylabel("Recall@10")
ax1.set_title("Recall vs nprobe"); ax1.grid(alpha=0.3)

ax2.plot(nprobe_values, latencies, "s-", color="#e67e22", lw=2, markersize=7)
ax2.set_xlabel("nprobe"); ax2.set_ylabel("Latency (ms) per 50 queries")
ax2.set_title("Latency vs nprobe"); ax2.grid(alpha=0.3)

ax3.plot(latencies, recalls, "^-", color="#9b59b6", lw=2, markersize=7)
for np_v, lat, rec in zip(nprobe_values, latencies, recalls):
    ax3.annotate(str(np_v), (lat, rec), textcoords="offset points",
                  xytext=(4, 3), fontsize=8)
ax3.set_xlabel("Latency (ms)"); ax3.set_ylabel("Recall@10")
ax3.set_title("Recall-latency frontier"); ax3.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nOptimal nprobe (recall ≥ 0.95):", 
      next((n for n, r in zip(nprobe_values, recalls) if r >= 0.95), "not reached"))


## 6. ID Mapping and Metadata Store

In [ ]:
# ── Production pattern: separate vector index from metadata ──────────
from dataclasses import dataclass, field

@dataclass
class Document:
    id:       str
    text:     str
    metadata: dict = field(default_factory=dict)


class VectorStore:
    """
    Vector index + metadata store.
    Keeps FAISS index for fast ANN, dict for metadata.
    """
    def __init__(self, embedder, dim: int, index_type: str = "flat"):
        self.embedder = embedder
        self.index    = VectorIndex(dim=dim, index_type=index_type)
        self._docs:  dict[int, Document] = {}
        self._next_id = 0

    def add_documents(self, docs: list[Document]):
        texts = [d.text for d in docs]
        vecs  = self.embedder.embed(texts)
        ids   = list(range(self._next_id, self._next_id + len(docs)))
        self.index.add(vecs, ids)
        for int_id, doc in zip(ids, docs):
            self._docs[int_id] = doc
        self._next_id += len(docs)

    def search(self, query: str, k: int = 5) -> list[tuple[Document, float]]:
        q_vec = self.embedder.embed([query])
        ids, scores = self.index.search(q_vec, k=k)
        results = []
        for int_id, score in zip(ids[0], scores[0]):
            if int_id >= 0 and int_id in self._docs:
                results.append((self._docs[int_id], float(score)))
        return results

    def __len__(self): return self._next_id


# ── Demo with a mini knowledge base ──────────────────────────────────
KB = [
    Document("d1", "Transformers use self-attention to process sequences in parallel.",
              {"category": "ml", "difficulty": "intermediate"}),
    Document("d2", "FAISS provides efficient similarity search over dense vectors.",
              {"category": "retrieval", "difficulty": "intermediate"}),
    Document("d3", "Gradient descent minimises the loss function iteratively.",
              {"category": "ml", "difficulty": "beginner"}),
    Document("d4", "HNSW builds a hierarchical proximity graph for ANN search.",
              {"category": "retrieval", "difficulty": "advanced"}),
    Document("d5", "Pasta carbonara is made with eggs, pecorino, guanciale and pepper.",
              {"category": "cooking", "difficulty": "beginner"}),
    Document("d6", "Rome was founded in 753 BC according to Roman tradition.",
              {"category": "history", "difficulty": "beginner"}),
    Document("d7", "Product quantisation compresses vectors for memory-efficient ANN.",
              {"category": "retrieval", "difficulty": "advanced"}),
    Document("d8", "Backpropagation computes gradients via the chain rule.",
              {"category": "ml", "difficulty": "intermediate"}),
]

store = VectorStore(embedder, dim=embedder.dim, index_type="flat")
store.add_documents(KB)
print(f"Vector store: {len(store)} documents indexed\n")

queries = [
    "how does attention work in transformers",
    "fast nearest neighbour search algorithms",
    "how do neural networks learn",
]
for q in queries:
    results = store.search(q, k=3)
    print(f"Q: {q}")
    for doc, score in results:
        print(f"  [{score:.3f}] [{doc.metadata.get('category','?')}] {doc.text[:60]}")
    print()


## 7. Recall@k and Evaluation Metrics

### Metrics for retrieval quality

**Recall@k** — fraction of relevant documents found in top-k:
$$\text{Recall@k} = \frac{|\text{retrieved}_k \cap \text{relevant}|}{|\text{relevant}|}$$

**Precision@k** — fraction of top-k that are relevant:
$$\text{Precision@k} = \frac{|\text{retrieved}_k \cap \text{relevant}|}{k}$$

**Mean Reciprocal Rank (MRR)** — position of the first relevant result:
$$\text{MRR} = \frac{1}{|Q|}\sum_{i=1}^{|Q|} \frac{1}{\text{rank}_i}$$

**NDCG@k** — normalised discounted cumulative gain, accounts for graded relevance.


In [ ]:
def recall_at_k(retrieved: list, relevant: set, k: int) -> float:
    return len(set(retrieved[:k]) & relevant) / max(len(relevant), 1)

def precision_at_k(retrieved: list, relevant: set, k: int) -> float:
    return len(set(retrieved[:k]) & relevant) / k

def mrr(retrieved_lists: list[list], relevant_sets: list[set]) -> float:
    rr = []
    for retrieved, relevant in zip(retrieved_lists, relevant_sets):
        for rank, item in enumerate(retrieved, 1):
            if item in relevant:
                rr.append(1 / rank); break
        else:
            rr.append(0.0)
    return float(np.mean(rr))

def ndcg_at_k(retrieved: list, relevant: set, k: int) -> float:
    dcg = sum(1 / math.log2(rank + 1)
              for rank, doc in enumerate(retrieved[:k], 1) if doc in relevant)
    ideal = sum(1 / math.log2(rank + 1) for rank in range(1, len(relevant) + 1))
    return dcg / ideal if ideal > 0 else 0.0


# ── Evaluate on small test set ────────────────────────────────────────
# Labelled test: (query, set of relevant doc ids)
labelled_queries = [
    ("attention mechanisms self-attention",   {"d1"}),
    ("vector similarity search nearest",      {"d2", "d4", "d7"}),
    ("training neural networks optimisation", {"d3", "d8"}),
]

k_values = [1, 3, 5]
print("Retrieval metrics on labelled test set:")
print(f"{'Metric':<12}", "  ".join(f"  @{k}" for k in k_values))
print("-" * 40)

all_retrieved = []
all_relevant  = []
for q_text, rel_ids in labelled_queries:
    results = store.search(q_text, k=max(k_values))
    ret_ids = [doc.id for doc, _ in results]
    all_retrieved.append(ret_ids)
    all_relevant.append(rel_ids)

for metric_name, fn in [("Recall",    recall_at_k),
                          ("Precision", precision_at_k),
                          ("NDCG",      ndcg_at_k)]:
    row = f"{metric_name:<12}"
    for k in k_values:
        vals = [fn(ret, rel, k) for ret, rel in zip(all_retrieved, all_relevant)]
        row += f"  {np.mean(vals):>5.3f}"
    print(row)

print(f"{'MRR':<12}  {mrr(all_retrieved, all_relevant):.3f}")


## 8. Engineering Insights

In [ ]:
# ── Scalability: query latency vs corpus size ─────────────────────────
import math

print("Theoretical query latency scaling (d=768, single CPU core):")
print(f"{'N vectors':>12} {'Brute-force':>14} {'IVF (nprobe=8)':>16} {'HNSW':>8}")
print("-" * 56)
for n in [1_000, 10_000, 100_000, 1_000_000, 10_000_000]:
    d = 768
    # Rough ops/sec estimate: ~1e9 float ops/sec on single CPU
    bf_ms   = n * d / 1e9 * 1000
    n_list  = int(4 * math.sqrt(n))
    ivf_ms  = (8 / n_list) * n * d / 1e9 * 1000 + 0.1   # probe 8 cells
    hnsw_ms = math.log2(n) * d / 1e9 * 1000 * 10         # ~10 layers
    print(f"{n:>12,} {bf_ms:>12.1f}ms {ivf_ms:>14.1f}ms {hnsw_ms:>6.2f}ms")

print()
print("Rule: HNSW for < 10M vectors; IVF-PQ for > 10M.")


## 9. Key Takeaways

| Concept | Summary |
|---|---|
| **Exact search** | $O(Nd)$ — correct, use for < 10K vectors |
| **IVF** | Partition + probe; nprobe controls recall/speed tradeoff |
| **HNSW** | Graph-based; best recall/speed in practice |
| **Recall@k** | Primary metric for ANN quality |
| **VectorStore** | Separate index (speed) from metadata store (richness) |
| **Index selection** | flat → IVF → IVFPQ as N grows |
| **nprobe tuning** | Sweep nprobe to find the recall ≥ 0.95 knee |

---
**Next notebook →** `03_vector_databases.ipynb`
